In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/ayyyyyyy/aqi_daily_data_top6_most_data.csv


In [3]:
# --- 1. Setup & Imports ---
# Install necessary libraries (run these in a notebook cell first!)
# !pip install torch torchvision torchaudio -q
# !pip install transformers datasets accelerate -q # If using Hugging Face's implementation
# !pip install pandas numpy scikit-learn matplotlib -q
# Add any other specific libraries needed for your chosen PatchTST implementation
# e.g., !pip install patchtst_package_name

import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn # For placeholder model & loss
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import os
import gc # Garbage collector
from tqdm.notebook import tqdm # Use notebook version for better display in Kaggle

print("--- Step 1: Setup & Imports Complete ---")

# --- 2. Configuration ---
class Config:
    # Data params
    # *** IMPORTANT: ADJUST PATH BELOW to match your Kaggle input ***
    data_path = "/kaggle/input/ayyyyyyy/aqi_daily_data_top6_most_data.csv" # <--- CHANGE THIS
    output_dir = "/kaggle/working/"
    date_col = 'ds'
    state_col = 'state'
    city_col = 'city'
    # Pollutants to forecast - should match columns in your CSV
    target_cols = ['PM2_5', 'PM10', 'NO', 'NO2', 'SO2', 'CO', 'NH3']
    freq = 'D' # Daily frequency

    # Model params (*** CRITICAL: ADJUST THESE FOR PATCHTST ***)
    context_length = 90    # How many past days the model sees (e.g., ~3 months)
    prediction_length = 90 # How many future days to predict at once (e.g., ~3 months)

    # --- PatchTST Specific Hyperparameters (Example Values - TUNE THESE) ---
    patch_length = 16      # Length of subsequences (patches)
    stride = 8             # Overlap between patches
    d_model = 128          # Model dimension (embedding size)
    n_heads = 8            # Number of attention heads
    num_encoder_layers = 3 # Number of transformer encoder layers
    dropout = 0.1          # Dropout rate
    # Add any other hyperparameters your specific PatchTST implementation requires
    # --- End PatchTST Specific ---

    # Training params
    batch_size = 32 # Adjust based on GPU memory
    epochs = 20     # Increase as needed, monitor validation loss
    learning_rate = 0.0001
    patience = 5    # For early stopping
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cfg = Config()
os.makedirs(cfg.output_dir, exist_ok=True)
print(f"Using device: {cfg.device}")
# Ensure target cols exist in list form
if not isinstance(cfg.target_cols, list): cfg.target_cols = list(cfg.target_cols)
print(f"Target columns: {cfg.target_cols}")
print("--- Step 2: Configuration Complete ---")

# --- 3. Load and Prepare Data ---
print("Loading data...")
try:
    df = pd.read_csv(cfg.data_path, parse_dates=[cfg.date_col])
    df = df.sort_values(by=[cfg.city_col, cfg.date_col]).reset_index(drop=True)
    print(f"Loaded data shape: {df.shape}")
    print(f"Cities in data: {df[cfg.city_col].unique()}")

    # Ensure all target columns exist
    missing_cols = [col for col in cfg.target_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Target columns missing from CSV: {missing_cols}")
    if cfg.city_col not in df.columns:
        raise ValueError(f"City column '{cfg.city_col}' not found.")
    if cfg.state_col not in df.columns:
        raise ValueError(f"State column '{cfg.state_col}' not found.")
    if not pd.api.types.is_datetime64_any_dtype(df[cfg.date_col]):
        raise ValueError(f"Date column '{cfg.date_col}' not correctly parsed as datetime.")

    # Basic Imputation (forward fill within each city group, then backward)
    print("Performing forward/backward fill for missing values...")
    df[cfg.target_cols] = df.groupby(cfg.city_col)[cfg.target_cols].ffill()
    df[cfg.target_cols] = df.groupby(cfg.city_col)[cfg.target_cols].bfill()

    # Check if NaNs remain (should only happen if a city has ALL NaNs for a column)
    if df[cfg.target_cols].isnull().any().any():
        print("Warning: NaNs remain after ffill/bfill. Filling with global median.")
        for col in cfg.target_cols:
             if df[col].isnull().any():
                  global_median = df[col].median()
                  print(f"  Filling NaNs in {col} with median: {global_median:.2f}")
                  df[col] = df[col].fillna(global_median)

    # Check final NaNs
    if df[cfg.target_cols].isnull().any().any():
         print("ERROR: NaNs still present after all filling attempts. Check data.")
         print(df[cfg.target_cols].isnull().sum())
    else:
         print("No NaNs remaining in target columns.")

except FileNotFoundError:
    print(f"ERROR: File not found at {cfg.data_path}. Please check the path in Configuration.")
    raise
except Exception as e:
    print(f"ERROR loading or processing data: {e}")
    raise

print("Data loading and initial preparation complete.")
print("--- Step 3: Load and Prepare Data Complete ---")


# --- 4. Feature Scaling ---
print("Scaling target features...")
scalers = {}
scaled_data = df.copy()

for city in df[cfg.city_col].unique():
    city_mask = scaled_data[cfg.city_col] == city
    if city_mask.sum() > 0: # Check if city has data
        scalers[city] = StandardScaler()
        # Fit and transform, making sure indices align
        scaled_values = scalers[city].fit_transform(scaled_data.loc[city_mask, cfg.target_cols])
        scaled_data.loc[city_mask, cfg.target_cols] = scaled_values
    else:
        print(f"Warning: No data found for city {city} during scaling.")

print("Data scaling complete.")
print("--- Step 4: Feature Scaling Complete ---")

# --- 5. Create Time Series Datasets ---
# *** IMPLEMENTATION REQUIRED *** - Tailor to your PatchTST model's input
class AQIDataset(Dataset):
    def __init__(self, data, cities, context_length, prediction_length, target_cols, date_col, city_col):
        self.context_length = context_length
        self.prediction_length = prediction_length
        self.target_cols = target_cols
        self.date_col = date_col
        self.city_col = city_col

        self.city_data = {city: group.reset_index(drop=True)
                          for city, group in data.groupby(city_col)}

        self.indices = []
        for city, city_df in self.city_data.items():
            if len(city_df) >= context_length + prediction_length:
                for i in range(len(city_df) - context_length - prediction_length + 1):
                    self.indices.append((city, i)) # Store (city, start_index)
            else:
                 print(f"Warning: City {city} has only {len(city_df)} data points, less than required {context_length + prediction_length}. Skipping.")


    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        city, start_idx = self.indices[idx]
        city_df = self.city_data[city]

        context_end = start_idx + self.context_length
        pred_end = context_end + self.prediction_length

        past_values = city_df.iloc[start_idx:context_end][self.target_cols].values
        future_values = city_df.iloc[context_end:pred_end][self.target_cols].values

        past_values_tensor = torch.tensor(past_values, dtype=torch.float32)
        future_values_tensor = torch.tensor(future_values, dtype=torch.float32)

        # *** Adapt the return dictionary based on your PatchTST model's expected input ***
        return {
            'past_values': past_values_tensor,
            'future_values': future_values_tensor # Label for training
            # 'city_id': city # Potentially add city info if needed by model embedding
        }

print("Dataset class defined.")
print("--- Step 5: Create Time Series Datasets Complete ---")


# --- 6. Split Data Chronologically ---
print("Splitting data chronologically...")
max_data_date = scaled_data[cfg.date_col].max()
print(f"Latest date in provided data: {max_data_date}")

# Use the last full year available for validation, the rest for training
val_year = max_data_date.year - 1 if max_data_date.month > 6 else max_data_date.year - 2 # Heuristic: if data ends early in year, use previous year for val
train_end_date = pd.Timestamp(f'{val_year-1}-12-31')
val_end_date = pd.Timestamp(f'{val_year}-12-31')

print(f"Train data up to: {train_end_date}")
print(f"Validation data from {train_end_date + pd.Timedelta(days=1)} to {val_end_date}")

train_data = scaled_data[scaled_data[cfg.date_col] <= train_end_date]
val_data = scaled_data[(scaled_data[cfg.date_col] > train_end_date) & (scaled_data[cfg.date_col] <= val_end_date)]

# Filter out cities that don't have enough data in both train and val periods
required_len = cfg.context_length + cfg.prediction_length
train_cities_valid = {city for city, group in train_data.groupby(cfg.city_col) if len(group) >= required_len}
val_cities_valid = {city for city, group in val_data.groupby(cfg.city_col) if len(group) >= required_len}

valid_cities = list(train_cities_valid & val_cities_valid)

if not valid_cities:
     raise ValueError("No cities have sufficient data in both training and validation periods after chronological split.")

print(f"Cities usable for training & validation: {valid_cities}")

# Re-filter data for valid cities only
train_data_filtered = train_data[train_data[cfg.city_col].isin(valid_cities)]
val_data_filtered = val_data[val_data[cfg.city_col].isin(valid_cities)]


train_dataset = AQIDataset(train_data_filtered, valid_cities, cfg.context_length, cfg.prediction_length, cfg.target_cols, cfg.date_col, cfg.city_col)
val_dataset = AQIDataset(val_data_filtered, valid_cities, cfg.context_length, cfg.prediction_length, cfg.target_cols, cfg.date_col, cfg.city_col)

# Check if datasets were created successfully
if len(train_dataset) == 0 or len(val_dataset) == 0:
    raise ValueError(f"Dataset creation resulted in zero samples (Train: {len(train_dataset)}, Val: {len(val_dataset)}). Check data splitting, lengths, and city data availability.")

train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True, num_workers=os.cpu_count()//2) # Use available cores
val_loader = DataLoader(val_dataset, batch_size=cfg.batch_size, shuffle=False, num_workers=os.cpu_count()//2)

print(f"Train samples: {len(train_dataset)}, Val samples: {len(val_dataset)}")
print("--- Step 6: Split Data Complete ---")


# --- 7. Define PatchTST Model ---
print("Defining PatchTST Model...")
# *** IMPLEMENTATION REQUIRED ***
# Replace this section with the actual PatchTST model definition.
# --- Start Placeholder (Simple LSTM - REPLACE) ---
class SimpleLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size, prediction_length, dropout):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.linear = nn.Linear(hidden_size, output_size * prediction_length) # Predict all steps at once
        self.prediction_length = prediction_length
        self.output_size = output_size

    def forward(self, past_values, **kwargs): # Match expected input keys from Dataset
        lstm_out, _ = self.lstm(past_values)
        last_time_step_out = lstm_out[:, -1, :]
        prediction_flat = self.linear(last_time_step_out)
        # Reshape to (batch, prediction_length, output_size)
        prediction = prediction_flat.view(past_values.size(0), self.prediction_length, self.output_size)
        return prediction

model = SimpleLSTM( # *** REPLACE THIS INSTANTIATION WITH YOUR PATCHTST MODEL ***
    input_size=len(cfg.target_cols),
    hidden_size=128, # Example
    num_layers=2,    # Example
    output_size=len(cfg.target_cols),
    prediction_length=cfg.prediction_length,
    dropout=cfg.dropout
)
# --- End Placeholder ---

model.to(cfg.device)
# Count parameters (optional)
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model defined (Placeholder Used!). Number of trainable parameters: {num_params:,}")
# print(model) # Optionally print the full model structure
print("--- Step 7: Define PatchTST Model Complete ---")


# --- 8. Define Loss Function and Optimizer ---
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=cfg.patience // 2, factor=0.1, verbose=True)

print("--- Step 8: Define Loss Function and Optimizer Complete ---")


# --- 9. Training Loop ---
print("Starting training...")
history = {'train_loss': [], 'val_loss': []}
best_val_loss = float('inf')
epochs_no_improve = 0

for epoch in range(cfg.epochs):
    model.train()
    epoch_train_loss = 0.0
    train_iterator = tqdm(train_loader, desc=f"Epoch {epoch+1}/{cfg.epochs} Training", leave=False)

    for i, batch in enumerate(train_iterator):
        try:
            optimizer.zero_grad()
            past_values = batch['past_values'].to(cfg.device)
            future_values = batch['future_values'].to(cfg.device)

            # *** Model Forward Pass ***
            outputs = model(past_values=past_values) # Adapt call signature if needed

            if outputs.shape != future_values.shape:
                print(f" Shape mismatch! Output: {outputs.shape}, Target: {future_values.shape}")
                # Try adjusting view if plausible, otherwise error
                try:
                    outputs = outputs.view(future_values.shape)
                except RuntimeError as e:
                     print(f" CRITICAL: Cannot reshape output to target. Check model output layer. Error: {e}")
                     raise e # Stop training if shapes fundamentally wrong


            loss = criterion(outputs, future_values)
            loss.backward()
            optimizer.step()

            epoch_train_loss += loss.item()
            train_iterator.set_postfix(loss=f"{loss.item():.4f}")

        except Exception as e:
            print(f"Error during training batch {i}: {e}")
            # Optionally skip batch or stop training
            continue # Skip batch

    avg_train_loss = epoch_train_loss / len(train_loader) if len(train_loader) > 0 else 0
    history['train_loss'].append(avg_train_loss)

    # --- Validation Loop ---
    model.eval()
    epoch_val_loss = 0.0
    val_iterator = tqdm(val_loader, desc=f"Epoch {epoch+1}/{cfg.epochs} Validation", leave=False)

    with torch.no_grad():
        for i, batch in enumerate(val_iterator):
            try:
                past_values = batch['past_values'].to(cfg.device)
                future_values = batch['future_values'].to(cfg.device)

                # *** Model Prediction ***
                predicted_values = model(past_values=past_values) # Adapt call signature if needed

                if predicted_values.shape != future_values.shape:
                    print(f" Shape mismatch (Val)! Output: {predicted_values.shape}, Target: {future_values.shape}")
                    try:
                       predicted_values = predicted_values.view(future_values.shape)
                    except RuntimeError as e:
                       print(f" CRITICAL (Val): Cannot reshape output to target. Error: {e}")
                       continue # Skip calculating loss for this batch


                loss = criterion(predicted_values, future_values)
                epoch_val_loss += loss.item()
                val_iterator.set_postfix(loss=f"{loss.item():.4f}")

            except Exception as e:
                 print(f"Error during validation batch {i}: {e}")
                 continue # Skip batch


    avg_val_loss = epoch_val_loss / len(val_loader) if len(val_loader) > 0 else float('inf')
    history['val_loss'].append(avg_val_loss)

    print(f"Epoch {epoch+1}/{cfg.epochs}: Train Loss = {avg_train_loss:.6f}, Val Loss = {avg_val_loss:.6f}")

    scheduler.step(avg_val_loss) # Update learning rate

    # --- Early Stopping & Model Checkpointing ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), os.path.join(cfg.output_dir, 'best_model.pth'))
        print(f"  Validation loss improved to {best_val_loss:.6f}. Saved best model.")
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        print(f"  Validation loss did not improve for {epochs_no_improve} epoch(s). Best: {best_val_loss:.6f}")

    if epochs_no_improve >= cfg.patience:
        print(f"Early stopping triggered after {epoch+1} epochs.")
        break

print("Training finished.")

# --- Load Best Model ---
best_model_path = os.path.join(cfg.output_dir, 'best_model.pth')
if os.path.exists(best_model_path):
    print(f"Loading best model from checkpoint (Best Val Loss: {best_val_loss:.6f})...")
    model.load_state_dict(torch.load(best_model_path))
    model.eval()
else:
    print("Warning: No best model checkpoint found. Using the model from the last epoch.")
    model.eval()


# --- Plot Training History ---
try:
    plt.figure(figsize=(10, 5))
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.title('Model Training History')
    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.legend()
    plt.grid(True)
    plot_path = os.path.join(cfg.output_dir, 'training_history.png')
    plt.savefig(plot_path)
    print(f"Training history plot saved to {plot_path}")
    # plt.show() # Uncomment to display in notebook
    plt.close() # Close plot to free memory
except Exception as plot_err:
    print(f"Could not save training plot: {plot_err}")


print("--- Step 9: Training Loop Complete ---")

# --- 10. Generate Future Forecasts (2024-2026) ---
print("\nGenerating future forecasts...")

# Determine forecast start date (day after the last date in the original dataset)
forecast_start_date = df[cfg.date_col].max() + pd.Timedelta(days=1)
forecast_end_date = pd.Timestamp('2026-12-31')
forecast_dates = pd.date_range(start=forecast_start_date, end=forecast_end_date, freq=cfg.freq)
print(f"Forecasting from {forecast_start_date.date()} to {forecast_end_date.date()} ({len(forecast_dates)} days)")

all_forecasts_list = []
cities_to_forecast = df[cfg.city_col].unique() # Use all cities from original filtered data

with torch.no_grad():
    for city in tqdm(cities_to_forecast, desc="Forecasting Cities"):
        print(f"  Generating forecast for: {city}")

        # Check if scaler exists for the city
        if city not in scalers:
            print(f"  Warning: Scaler not found for {city}. Skipping forecast.")
            continue

        # Get the last 'context_length' days of REAL data for this city (use scaled data)
        last_real_data = scaled_data[(scaled_data[cfg.city_col] == city)].tail(cfg.context_length)

        if len(last_real_data) < cfg.context_length:
            print(f"  Warning: Not enough historical data ({len(last_real_data)} points) for {city} to start forecast. Skipping.")
            continue

        # Initialize context window
        current_context_values = torch.tensor(last_real_data[cfg.target_cols].values, dtype=torch.float32).unsqueeze(0).to(cfg.device)
        # Add other features if your model uses them

        city_predictions_scaled = [] # Store scaled predictions

        # Calculate how many prediction steps are needed
        total_pred_len = len(forecast_dates)
        num_iterations = (total_pred_len + cfg.prediction_length - 1) // cfg.prediction_length

        iter_range = tqdm(range(num_iterations), desc=f"  Predicting Steps for {city}", leave=False)
        for i in iter_range:
            try:
                # *** Generate Prediction using the loaded best model ***
                prediction_output = model(past_values=current_context_values) # Adapt call signature
                # Output shape: [1, prediction_length, num_targets]

                # Detach, move to CPU, convert to numpy
                prediction_scaled_np = prediction_output.squeeze(0).cpu().numpy() # Shape: [prediction_length, num_targets]

                # Store the SCALED prediction
                city_predictions_scaled.append(prediction_scaled_np)

                # --- Update context for the next iteration ---
                # Use the SCALED prediction to update the context
                new_block_tensor = torch.tensor(prediction_scaled_np, dtype=torch.float32).to(cfg.device)

                # Roll the window:
                # Take last (context_length - prediction_length) from old context + new prediction block
                if cfg.prediction_length <= cfg.context_length:
                    # Take last part of old context values
                    old_context_part = current_context_values[:, - (cfg.context_length - cfg.prediction_length):, :]
                    # Concatenate
                    new_context_tensor = torch.cat((old_context_part, new_block_tensor.unsqueeze(0)), dim=1)
                else:
                    # If predicting longer than context, just use the latest prediction block, taking the last context_length part
                    new_context_tensor = new_block_tensor[-cfg.context_length:, :].unsqueeze(0)

                current_context_values = new_context_tensor
                # Update time features for the new context window if needed

            except Exception as pred_err:
                 print(f"  ERROR during prediction step {i} for {city}: {pred_err}")
                 # Option: break the loop for this city, or fill with NaNs, etc.
                 city_predictions_scaled.append(np.full((cfg.prediction_length, len(cfg.target_cols)), np.nan)) # Fill failed step with NaNs
                 # Need robust context update if errors occur
                 break # Stop predicting for this city on error


        # Combine all prediction steps for the city
        if city_predictions_scaled:
            full_city_forecast_scaled = np.concatenate(city_predictions_scaled, axis=0)
            # Trim to the exact number of forecast dates needed
            full_city_forecast_scaled = full_city_forecast_scaled[:len(forecast_dates)]

            # Check for NaNs introduced by errors
            if np.isnan(full_city_forecast_scaled).any():
                print(f"  Warning: NaNs found in forecast for {city}, potentially due to prediction errors.")
                # Option: Fill NaNs (e.g., ffill) or leave them
                # temp_df = pd.DataFrame(full_city_forecast_scaled).ffill().bfill()
                # full_city_forecast_scaled = temp_df.values

            # Inverse transform using the city's scaler
            try:
                 full_city_forecast_rescaled = scalers[city].inverse_transform(full_city_forecast_scaled)
            except Exception as scale_err:
                 print(f"  ERROR inverse scaling forecast for {city}: {scale_err}")
                 continue # Skip this city if scaling fails


            # Create DataFrame for this city's forecast
            forecast_df = pd.DataFrame(full_city_forecast_rescaled, columns=cfg.target_cols)
            forecast_df[cfg.date_col] = forecast_dates
            forecast_df[cfg.city_col] = city
            # Add state info
            try:
                state = df[df[cfg.city_col] == city][cfg.state_col].iloc[0] # Get state from original df
                forecast_df[cfg.state_col] = state
            except IndexError:
                print(f"  Warning: Could not find state for city {city} in original data.")
                forecast_df[cfg.state_col] = "Unknown" # Or handle as needed

            all_forecasts_list.append(forecast_df)
        else:
             print(f"  No predictions generated for {city}.")


if not all_forecasts_list:
     raise ValueError("No forecasts were generated for any city. Check prediction loop and data.")

final_forecast_df = pd.concat(all_forecasts_list, ignore_index=True)
# Clip negative predictions after inverse scaling
for col in cfg.target_cols:
     if col not in ['CO']: # Allow CO potentially slightly negative if model predicts it
         final_forecast_df[col] = final_forecast_df[col].clip(lower=0)

print(f"Generated forecast shape: {final_forecast_df.shape}")
print("--- Step 10: Generate Future Forecasts Complete ---")


# --- 11. Calculate AQI for Forecasts ---
print("\nCalculating AQI for forecasts...")

# --- CPCB AQI Calculation Logic (Copied again for clarity) ---
# (Ensure breakpoints and pollutants_to_consider_aqi are defined)
breakpoints = {
    'PM2_5': [ (0, 30, 0, 50), (31, 60, 51, 100), (61, 90, 101, 200), (91, 120, 201, 300), (121, 250, 301, 400), (251, float('inf'), 401, 500)],
    'PM10': [ (0, 50, 0, 50), (51, 100, 51, 100), (101, 250, 101, 200), (251, 350, 201, 300), (351, 430, 301, 400), (431, float('inf'), 401, 500)],
    'SO2': [ (0, 40, 0, 50), (41, 80, 51, 100), (81, 380, 101, 200), (381, 800, 201, 300), (801, 1600, 301, 400), (1601, float('inf'), 401, 500)],
    'NO2': [ (0, 40, 0, 50), (41, 80, 51, 100), (81, 180, 101, 200), (181, 280, 201, 300), (281, 400, 301, 400), (401, float('inf'), 401, 500)],
    'CO': [ (0, 1.0, 0, 50), (1.1, 2.0, 51, 100), (2.1, 10.0, 101, 200), (10.1, 17.0, 201, 300), (17.1, 34.0, 301, 400), (34.1, float('inf'), 401, 500)],
    'NH3': [ (0, 200, 0, 50), (201, 400, 51, 100), (401, 800, 101, 200), (801, 1200, 201, 300), (1201, 1800, 301, 400), (1801, float('inf'), 401, 500)]
}
pollutants_to_consider_aqi = [p for p in cfg.target_cols if p in breakpoints and p in final_forecast_df.columns]

def calculate_sub_index_pred(pollutant, concentration):
    if pd.isna(concentration): return np.nan
    bp_list = breakpoints.get(pollutant)
    if not bp_list: return np.nan
    concentration = float(concentration)
    bp = None
    for conc_low, conc_high, aqi_low, aqi_high in bp_list:
        if conc_low == 0:
             if concentration >= conc_low and concentration <= conc_high: bp = (conc_low, conc_high, aqi_low, aqi_high); break
        else:
             if concentration > conc_low and concentration <= conc_high: bp = (conc_low, conc_high, aqi_low, aqi_high); break
    if bp is None and concentration > bp_list[-1][1]: bp = bp_list[-1] # Use highest range if above max
    if bp is None and concentration < bp_list[0][0]: bp = bp_list[0] # Use lowest range if below min (e.g., negative CO)
    if bp is None: return np.nan # Should not happen if clipping handled
    conc_low, conc_high, aqi_low, aqi_high = bp
    if conc_high == conc_low: sub_index = aqi_low
    else: sub_index = ((aqi_high - aqi_low) / (conc_high - conc_low)) * (concentration - conc_low) + aqi_low
    return min(round(sub_index), 500) # Cap at 500

def calculate_overall_aqi_pred(row):
    max_sub_index = 0; dominant_pollutant = 'N/A'; calculated_any = False
    available_pollutants_row = [p for p in pollutants_to_consider_aqi if p in row.index]
    for pollutant in available_pollutants_row:
        sub_index = calculate_sub_index_pred(pollutant, row[pollutant])
        if not pd.isna(sub_index):
            calculated_any = True
            if sub_index >= max_sub_index: # >= ensures we capture the last dominant one if tied
                max_sub_index = sub_index
                dominant_pollutant = pollutant
    if not calculated_any: return pd.Series([np.nan, 'N/A'], index=['AQI', 'Dominant_Pollutant'])
    else: return pd.Series([int(max_sub_index), dominant_pollutant], index=['AQI', 'Dominant_Pollutant']) # Return AQI as integer
# --- End AQI Logic ---

# Apply AQI calculation row-wise using progress_apply
tqdm.pandas(desc="Calculating AQI on Forecasts")
aqi_forecast_results = final_forecast_df.progress_apply(calculate_overall_aqi_pred, axis=1)
final_forecast_df = pd.concat([final_forecast_df, aqi_forecast_results], axis=1)

# Handle potential NaNs in AQI
final_forecast_df['AQI'] = final_forecast_df['AQI'].fillna(-1).astype(int) # Use -1 for missing AQI
final_forecast_df['Dominant_Pollutant'] = final_forecast_df['Dominant_Pollutant'].fillna('N/A')

print("AQI calculation for forecasts complete.")
print(final_forecast_df[[cfg.date_col, cfg.city_col, 'AQI', 'Dominant_Pollutant']].head())
print("--- Step 11: Calculate AQI for Forecasts Complete ---")


# --- 12. Save Final Predictions ---
forecast_filename = os.path.join(cfg.output_dir, "predicted_aqi_top6_2024_2026.csv")

# Select and order columns for the final CSV
output_cols = [cfg.date_col, cfg.state_col, cfg.city_col] + cfg.target_cols + ['AQI', 'Dominant_Pollutant']
# Ensure all columns exist before selecting
output_cols = [col for col in output_cols if col in final_forecast_df.columns]
final_save_df = final_forecast_df[output_cols]

final_save_df.to_csv(forecast_filename, index=False, float_format='%.2f') # Save with 2 decimal places
print(f"\nFinal forecasts saved to: {forecast_filename}")
print(f"Final saved shape: {final_save_df.shape}")
print("\nFirst 5 rows of saved data:")
print(final_save_df.head())
print("--- Step 12: Save Final Predictions Complete ---")


# --- Optional: Clean up memory ---
del df, scaled_data, train_data, val_data, train_dataset, val_dataset, train_loader, val_loader, final_forecast_df, aqi_forecast_results, final_save_df
# Keep model loaded if you want to inspect or save manually outside the loop
gc.collect()
if cfg.device == torch.device("cuda"):
    torch.cuda.empty_cache()
print("--- Cleanup Complete ---")



--- Step 1: Setup & Imports Complete ---
Using device: cuda
Target columns: ['PM2_5', 'PM10', 'NO', 'NO2', 'SO2', 'CO', 'NH3']
--- Step 2: Configuration Complete ---
Loading data...
Loaded data shape: (28694, 12)
Cities in data: ['Bengaluru' 'Chennai' 'Delhi' 'Hyderabad' 'Lucknow' 'Mumbai']
Performing forward/backward fill for missing values...
No NaNs remaining in target columns.
Data loading and initial preparation complete.
--- Step 3: Load and Prepare Data Complete ---
Scaling target features...
Data scaling complete.
--- Step 4: Feature Scaling Complete ---
Dataset class defined.
--- Step 5: Create Time Series Datasets Complete ---
Splitting data chronologically...
Latest date in provided data: 2023-03-31 00:00:00
Train data up to: 2020-12-31 00:00:00
Validation data from 2021-01-01 00:00:00 to 2021-12-31 00:00:00
Cities usable for training & validation: ['Bengaluru', 'Lucknow', 'Mumbai', 'Hyderabad', 'Chennai', 'Delhi']
Train samples: 22700, Val samples: 1116
--- Step 6: Split Da

/usr/local/lib/python3.11/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 1/20 Training:   0%|          | 0/710 [00:00<?, ?it/s]

Epoch 1/20 Validation:   0%|          | 0/35 [00:00<?, ?it/s]

Epoch 1/20: Train Loss = 0.804341, Val Loss = 0.774061
  Validation loss improved to 0.774061. Saved best model.


...

  Generating forecast for: Mumbai


  Predicting Steps for Mumbai:   0%|          | 0/16 [00:00<?, ?it/s]

Generated forecast shape: (8226, 10)
--- Step 10: Generate Future Forecasts Complete ---

Calculating AQI for forecasts...


Calculating AQI on Forecasts:   0%|          | 0/8226 [00:00<?, ?it/s]

AQI calculation for forecasts complete.
          ds       city  AQI Dominant_Pollutant
0 2023-04-01  Bengaluru   70               PM10
1 2023-04-02  Bengaluru   70               PM10
2 2023-04-03  Bengaluru   70               PM10
3 2023-04-04  Bengaluru   69               PM10
4 2023-04-05  Bengaluru   70               PM10
--- Step 11: Calculate AQI for Forecasts Complete ---

Final forecasts saved to: /kaggle/working/predicted_aqi_top6_2024_2026.csv
Final saved shape: (8226, 12)

First 5 rows of saved data:
          ds      state       city      PM2_5       PM10        NO        NO2  \
0 2023-04-01  Karnataka  Bengaluru  31.347538  70.187195  6.622480  21.329638   
1 2023-04-02  Karnataka  Bengaluru  31.509888  69.952576  6.722170  21.966169   
2 2023-04-03  Karnataka  Bengaluru  31.906986  69.888412  6.474695  21.628004   
3 2023-04-04  Karnataka  Bengaluru  31.474094  69.282814  6.750934  21.890629   
4 2023-04-05  Karnataka  Bengaluru  30.918463  69.780876  6.401180  21.643911 